# GPU • ONNX et TensorRT

ResNet-18, 50 images du split de test CIFAR-10 et lots de 10. Trois exécutions dans des processus Python distincts.

**Runtime Colab** · GPU NVIDIA.
Ce notebook prépare les artefacts communs. Lancer celui-ci en premier.

In [1]:
%pip -q install "onnx>=1.18,<2" "tensorrt-cu12>=10.13,<11" nvidia-ml-py

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/TP1_ResNet18")
ROOT.mkdir(parents=True, exist_ok=True)
CAMPAIGN = "colab-tp1"  # Même valeur dans les quatre notebooks.
ENV = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
SCRIPT = Path("/content/tp1_benchmark.py")

def executer(*args):
    subprocess.run([sys.executable, str(SCRIPT), str(ROOT), *map(str, args)],
                   env=ENV, check=True)

Mounted at /content/drive


## Implémentation du pipeline

        Les trois processus rechargent le modèle et exécutent le même protocole. Les fichiers JSON
        contiennent les mesures, les prédictions et la configuration matérielle.

In [4]:
%%writefile /content/tp1_benchmark.py
import argparse
import hashlib
import json
import platform
import re
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
import torchvision
from torchvision.models import ResNet18_Weights, resnet18

BATCH = 10
THREADS = 2
WARMUP = 5
PASSES = 10
RUNS = 3
DIGESTS = ("weights.pt", "test_data.pt", "model.onnx")
SCOPE = "Entrée CPU prétraitée → logits CPU, transferts inclus"
WEIGHTS = ResNet18_Weights.IMAGENET1K_V1


def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def prepare(root):
    import onnx
    from torch.utils.data import DataLoader, Subset

    if not torch.cuda.is_available():
        raise RuntimeError("Activer un GPU NVIDIA dans Colab pour la préparation.")
    generator = torch.Generator().manual_seed(42)
    torch.manual_seed(42)
    torch.backends.cudnn.allow_tf32 = False
    transform = WEIGHTS.transforms()
    train = torchvision.datasets.CIFAR10("/content/data", train=True, download=True, transform=transform)
    test = torchvision.datasets.CIFAR10("/content/data", train=False, download=True, transform=transform)
    train_indices = np.random.default_rng(42).choice(len(train), 5000, replace=False)
    indices = np.random.default_rng(42).choice(len(test), 50, replace=False).tolist()
    loader = DataLoader(Subset(train, train_indices.tolist()), batch_size=64, num_workers=2)
    model = resnet18(weights=WEIGHTS).eval().cuda()
    features, targets = [], []
    backbone = torch.nn.Sequential(*list(model.children())[:-1])
    with torch.no_grad():
        for images, labels in loader:
            features.append(backbone(images.cuda()).flatten(1))
            targets.append(labels.cuda())
    features, targets = torch.cat(features), torch.cat(targets)
    model.fc = torch.nn.Linear(model.fc.in_features, 10).cuda()
    optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
    for epoch in range(2):
        for idx in torch.randperm(len(train_indices), generator=generator).split(64):
            x, y = features[idx], targets[idx]
            optimizer.zero_grad(set_to_none=True)
            torch.nn.functional.cross_entropy(model.fc(x), y).backward()
            optimizer.step()
        print(f"Tête CIFAR-10, époque {epoch + 1}/2", flush=True)
    model = model.eval().cpu()
    images, labels = zip(*(test[i] for i in indices))
    torch.save({"X": torch.stack(images), "Y": torch.tensor(labels), "indices": indices,
                "classes": test.classes}, root / "test_data.pt")
    torch.save(model.state_dict(), root / "weights.pt")
    with torch.no_grad():
        torch.onnx.export(model, torch.zeros(BATCH, 3, 224, 224), str(root / "model.onnx"),
                          input_names=["input"], output_names=["output"], opset_version=17,
                          dynamo=False)
    onnx.checker.check_model(str(root / "model.onnx"))
    manifest = {"files": {name: sha256(root / name) for name in DIGESTS},
                "torch": str(torch.__version__), "torchvision": torchvision.__version__,
                "onnx": onnx.__version__, "training_seed": 42,
                "test_indices": indices, "classes": test.classes,
                "preprocessing": str(transform)}
    write_json(root / "manifest.json", manifest)
    print("Préparation validée", flush=True)


def write_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    tmp.replace(path)


def tensorrt_predictor(root):
    import tensorrt as trt

    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    if not parser.parse_from_file(str(root / "model.onnx")):
        raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    config.clear_flag(trt.BuilderFlag.TF32)
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("Échec de construction du moteur TensorRT.")
    runtime = trt.Runtime(logger)
    engine = runtime.deserialize_cuda_engine(serialized)
    if engine is None:
        raise RuntimeError("Échec de désérialisation TensorRT.")
    context = engine.create_execution_context()
    if context is None:
        raise RuntimeError("Échec de création du contexte TensorRT.")
    for name, shape in (("input", (BATCH, 3, 224, 224)), ("output", (BATCH, 10))):
        if tuple(engine.get_tensor_shape(name)) != shape or engine.get_tensor_dtype(name) != trt.float32:
            raise RuntimeError(f"Format inattendu pour le tenseur {name}.")
    out = torch.empty((BATCH, 10), device="cuda", dtype=torch.float32)
    if not context.set_tensor_address("output", out.data_ptr()):
        raise RuntimeError("Impossible de lier la sortie TensorRT.")

    def predict(batch):
        x = batch.to("cuda")
        if not context.set_tensor_address("input", x.data_ptr()):
            raise RuntimeError("Impossible de lier l’entrée TensorRT.")
        if not context.execute_async_v3(torch.cuda.current_stream().cuda_stream):
            raise RuntimeError("Échec de l’inférence TensorRT.")
        return out.cpu().numpy()

    # Le moteur, le runtime et le logger restent vivants jusqu’à la fin des inférences.
    return predict, trt.__version__, (logger, runtime, engine)


def energy_reader(gpu):
    if gpu:
        try:
            import pynvml as nv
        except ImportError:
            return None, None
        try:
            nv.nvmlInit()
            props = torch.cuda.get_device_properties(0)
            uuid = getattr(props, "uuid", None)
            if nv.nvmlDeviceGetCount() == 1:
                handle = nv.nvmlDeviceGetHandleByIndex(0)
            elif uuid is not None:
                handle = nv.nvmlDeviceGetHandleByUUID(str(uuid))
            else:
                return None, None
            reader = lambda: nv.nvmlDeviceGetTotalEnergyConsumption(handle) / 1e3
            reader()
            return "GPU NVML", reader
        except (OSError, nv.NVMLError):
            return None, None
    zones = list(Path("/sys/class/powercap").glob("intel-rapl:[0-9]*"))
    try:
        packages = [p for p in zones if re.fullmatch(r"intel-rapl:\d+", p.name)
                    and (p / "name").read_text().strip().startswith("package")]
        if len(packages) == 1:
            package = packages[0]
            reader = lambda: int((package / "energy_uj").read_text()) / 1e6
            reader()
            return "Package CPU RAPL", reader
    except (OSError, ValueError):
        pass
    return None, None


def energy_value(reader):
    try:
        return reader() if reader else None
    except Exception:
        # Une erreur du compteur ne doit pas faire perdre les mesures de latence.
        return None


def cpu_name():
    if Path("/proc/cpuinfo").exists():
        match = re.search(r"^model name\s*:\s*(.+)$", Path("/proc/cpuinfo").read_text(), re.M)
        if match:
            return match.group(1)
    return platform.processor() or platform.machine()


def benchmark(root, pipeline, run, campaign, colab, launch_group):
    torch.set_num_threads(THREADS)
    torch.set_num_interop_threads(1)
    gpu = pipeline.startswith("gpu_")
    if gpu:
        if not torch.cuda.is_available():
            raise RuntimeError("Activer un GPU NVIDIA dans Colab.")
        torch.backends.cudnn.allow_tf32 = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.benchmark = False
    manifest = json.loads((root / "manifest.json").read_text())
    for name in DIGESTS:
        if sha256(root / name) != manifest["files"][name]:
            raise RuntimeError(f"Fichier commun modifié ou incomplet : {name}")
    data = torch.load(root / "test_data.pt", map_location="cpu", weights_only=True)
    if data["X"].shape != (50, 3, 224, 224) or data["Y"].shape != (50,):
        raise RuntimeError("Les données d’évaluation doivent contenir 50 images.")
    batches, labels = data["X"].split(BATCH), data["Y"].numpy()
    predict, version, trt_resources = tensorrt_predictor(root)
    source, reader = energy_reader(gpu)
    with torch.inference_mode():
        for i in range(WARMUP):
            predict(batches[i % len(batches)])
        if gpu:
            torch.cuda.synchronize()
        durations = []
        started_energy = energy_value(reader)
        expected = None
        for _ in range(PASSES):
            outputs = []
            for batch in batches:
                start = perf_counter()
                logits = predict(batch)
                durations.append(perf_counter() - start)
                outputs.append(logits)
            logits = np.concatenate(outputs)
            if not np.isfinite(logits).all():
                raise RuntimeError("Logits non finis.")
            predictions = logits.argmax(axis=1)
            if expected is None:
                expected = predictions
            elif not np.array_equal(predictions, expected):
                raise RuntimeError("Prédictions variables pendant cette exécution.")
        ended_energy = energy_value(reader)
    energy = None
    if started_energy is not None and ended_energy is not None and ended_energy > started_energy:
        energy = ended_energy - started_energy
    count = len(labels) * PASSES
    result = {
        "pipeline": pipeline, "run": run, "campaign": campaign, "colab": colab,
        "launch_group": launch_group,
        "manifest": manifest["files"], "indices": data["indices"],
        "labels": labels.tolist(), "predictions": expected.tolist(),
        "metrics": {"debit_images_s": count / sum(durations),
                    "latence_ms_lot": 1000 * float(np.mean(durations)),
                    "top1_pct": 100 * float(np.mean(expected == labels)),
                    "energie_j": energy, "energie_j_image": energy / count if energy else None},
        "metadata": {"cpu": cpu_name(), "gpu": torch.cuda.get_device_name(0) if gpu else None,
                     "python": platform.python_version(), "torch": str(torch.__version__),
                     "torchvision": torchvision.__version__, "runtime": version,
                     "cuda": torch.version.cuda if gpu else None,
                     "cudnn": torch.backends.cudnn.version() if gpu else None,
                     "threads": THREADS, "batch": BATCH, "passes": PASSES, "warmup": WARMUP,
                     "precision": "FP32 (TF32 désactivé sur GPU)", "scope": SCOPE,
                     "energy_sensor": source}
    }
    write_json(root / "mesures" / campaign / f"{pipeline}_{run}.json", result)
    print(pipeline, run, result["metrics"], flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("root", type=Path)
    parser.add_argument("pipeline", choices=('prepare', 'gpu_tensorrt'))
    parser.add_argument("--run", type=int, choices=range(1, RUNS + 1))
    parser.add_argument("--campaign", default="colab-tp1")
    parser.add_argument("--colab", action="store_true")
    parser.add_argument("--groupe", default=None)
    args = parser.parse_args()
    if args.pipeline == "prepare":
        prepare(args.root)
    else:
        if args.run is None or not re.fullmatch(r"[A-Za-z0-9_-]+", args.campaign):
            parser.error("Indiquer un numéro d’exécution et un nom de campagne valide.")
        (args.root / "mesures" / args.campaign).mkdir(parents=True, exist_ok=True)
        benchmark(args.root, args.pipeline, args.run, args.campaign, args.colab, args.groupe)

Writing /content/tp1_benchmark.py


In [8]:
import subprocess
import sys

if (ROOT / "manifest.json").exists():
    print("Artefacts communs déjà préparés dans", ROOT)
else:
    # Afficher la progression pendant l’extraction des features.
    code = SCRIPT.read_text(encoding="utf-8")
    boucle = "        for images, labels in loader:\n"
    if boucle in code:
        code = code.replace(
            boucle,
            "        for i, (images, labels) in enumerate(loader, 1):\n"
            "            if i == 1 or i % 10 == 0 or i == len(loader):\n"
            '                print(f"Extraction ResNet-18 : lot {i}/{len(loader)}", flush=True)\n',
            1,
        )
        SCRIPT.write_text(code, encoding="utf-8")

    print("Démarrage de la préparation. Vérification et chargement des données…")
    cmd = [sys.executable, "-u", str(SCRIPT), str(ROOT), "prepare"]
    with subprocess.Popen(
        cmd, env=ENV, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    ) as process:
        for ligne in process.stdout:
            print(ligne.rstrip(), flush=True)
        if process.wait() != 0:
            raise RuntimeError("La préparation a échoué. Voir l’erreur affichée ci-dessus.")

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  4%|▍         | 6.75M/170M [01:04<45:37, 59.8kB/s]
  4%|▍         | 6.78M/170M [01:04<39:17, 69.4kB/s]
  4%|▍         | 6.82M/170M [01:04<37:05, 73.5kB/s]
  4%|▍         | 6.85M/170M [01:04<32:05, 85.0kB/s]
  4%|▍         | 6.88M/170M [01:05<28:01, 97.3kB/s]
  4%|▍         | 6.91M/170M [01:05<24:58, 109kB/s]
  4%|▍         | 6.95M/170M [01:05<22:38, 120kB/s]
  4%|▍         | 7.01M/170M [01:05<16:53, 161kB/s]
  4%|▍         | 7.05M/170M [01:06<17:23, 157kB/s]
  4%|▍         | 7.08M/170M [01:06<18:53, 144kB/s]
  4%|▍         | 7.11M/170M [01:06<20:06, 135kB/s]
  4%|▍         | 7.14M/170M [01:06<20:57, 130kB/s]
  4%|▍         | 7.18M/170M [01:07<23:14, 117kB/s]
  4%|▍         | 7.21M/170M [01:07<23:14, 117kB/s]
  4%|▍         | 7.24M/170M [01:07<23:21, 117kB/s]
  4%|▍         | 7.27M/170M [01:08<23:23, 116kB/s]
  4%|▍         | 7.31M/170M [01:08<23:20, 117kB/s]
  4%|▍         | 7.34M/170M [01:08<23:22, 116kB/s]

## Mesures et statistiques

In [9]:
import json
import uuid
import pandas as pd

if not (ROOT / "manifest.json").exists():
    raise FileNotFoundError("Exécuter d’abord la préparation avec le notebook 03.")
PIPELINE = 'gpu_tensorrt'
inputs = ROOT / "mesures" / CAMPAIGN
groupe = uuid.uuid4().hex
for numero in range(1, 4):
    executer(PIPELINE, "--campaign", CAMPAIGN, "--run", numero, "--colab", "--groupe", groupe)
mesures = [json.loads((inputs / f"{PIPELINE}_{i}.json").read_text()) for i in range(1, 4)]
df = pd.DataFrame([r["metrics"] for r in mesures], dtype=float)
print("Environnement :", mesures[-1]["metadata"])
display(df.rename_axis("exécution").set_axis(range(1, 4)))
display(df.agg(["count", "mean", "std", "min", "max"]))

Environnement : {'cpu': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'gpu': 'Tesla T4', 'python': '3.13.16', 'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'runtime': '10.16.1.11', 'cuda': '13.0', 'cudnn': 92800, 'threads': 2, 'batch': 10, 'passes': 10, 'warmup': 5, 'precision': 'FP32 (TF32 désactivé sur GPU)', 'scope': 'Entrée CPU prétraitée → logits CPU, transferts inclus', 'energy_sensor': 'GPU NVML'}


,debit_images_s,latence_ms_lot,top1_pct,energie_j,energie_j_image
1,1157.749691,8.637446,76.0,34.388,0.068776
2,1152.633961,8.675781,76.0,34.408,0.068816
3,1142.141875,8.755480,76.0,34.366,0.068732


,debit_images_s,latence_ms_lot,top1_pct,energie_j,energie_j_image
count,3.000000,3.000000,3.0,3.000000,3.000000
mean,1150.841842,8.689569,76.0,34.387333,0.068775
std,7.956742,0.060213,0.0,0.021008,0.000042
min,1142.141875,8.637446,76.0,34.366000,0.068732
max,1157.749691,8.755480,76.0,34.408000,0.068816
